In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/JoshTalks_ASR"
OUTPUT_DIR = os.path.join(BASE_DIR, "05_outputs", "baseline_eval")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Baseline output folder ready:", OUTPUT_DIR)

Baseline output folder ready: /content/drive/MyDrive/JoshTalks_ASR/05_outputs/baseline_eval


In [ ]:
!pip install -q datasets transformers evaluate jiwer librosa soundfile

In [ ]:
import os
import pandas as pd
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate
from tqdm import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [1]:
from datasets import load_dataset_builder

builder = load_dataset_builder("google/fleurs", "hi_in")
print(builder.info)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

fleurs.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found fleurs.py

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import list_repo_files

files = list_repo_files("google/fleurs", repo_type="dataset")
print("Total files:", len(files))
print(files[:50])


Total files: 616
['.gitattributes', 'README.md', 'data/af_za/audio/dev.tar.gz', 'data/af_za/audio/test.tar.gz', 'data/af_za/audio/train.tar.gz', 'data/af_za/dev.tsv', 'data/af_za/test.tsv', 'data/af_za/train.tsv', 'data/am_et/audio/dev.tar.gz', 'data/am_et/audio/test.tar.gz', 'data/am_et/audio/train.tar.gz', 'data/am_et/dev.tsv', 'data/am_et/test.tsv', 'data/am_et/train.tsv', 'data/ar_eg/audio/dev.tar.gz', 'data/ar_eg/audio/test.tar.gz', 'data/ar_eg/audio/train.tar.gz', 'data/ar_eg/dev.tsv', 'data/ar_eg/test.tsv', 'data/ar_eg/train.tsv', 'data/as_in/audio/dev.tar.gz', 'data/as_in/audio/test.tar.gz', 'data/as_in/audio/train.tar.gz', 'data/as_in/dev.tsv', 'data/as_in/test.tsv', 'data/as_in/train.tsv', 'data/ast_es/audio/dev.tar.gz', 'data/ast_es/audio/test.tar.gz', 'data/ast_es/audio/train.tar.gz', 'data/ast_es/dev.tsv', 'data/ast_es/test.tsv', 'data/ast_es/train.tsv', 'data/az_az/audio/dev.tar.gz', 'data/az_az/audio/test.tar.gz', 'data/az_az/audio/train.tar.gz', 'data/az_az/dev.tsv', 'd

In [ ]:
hi_files = [f for f in files if "data/hi_in/" in f]
print("Total hi_in files:", len(hi_files))
for f in hi_files:
    print(f)

Total hi_in files: 6
data/hi_in/audio/dev.tar.gz
data/hi_in/audio/test.tar.gz
data/hi_in/audio/train.tar.gz
data/hi_in/dev.tsv
data/hi_in/test.tsv
data/hi_in/train.tsv


In [ ]:
from huggingface_hub import hf_hub_download

test_tsv_path = hf_hub_download(
    repo_id="google/fleurs",
    filename="data/hi_in/test.tsv",
    repo_type="dataset"
)

print("Downloaded TSV path:")
print(test_tsv_path)

test.tsv: 0.00B [00:00, ?B/s]

Downloaded TSV path:
/root/.cache/huggingface/hub/datasets--google--fleurs/snapshots/d7c758a6dceecd54a98cac43404d3d576e721f07/data/hi_in/test.tsv


In [ ]:
test_df = pd.read_csv(test_tsv_path, sep="\t")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
test_df.head()

Shape: (417, 7)
Columns: ['1839', '7251883826251297781.wav', 'स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा) मार्ग जैसा ही सोचें।', 'स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा मार्ग जैसा ही सोचें।', 'स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई क ि ं ग | ल ं ब ी | प ै द ल | य ा त ् र ा | म ा र ् ग | ज ै स ा | ह ी | स ो च े ं । |', '149760', 'FEMALE']


,1839,7251883826251297781.wav,स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा) मार्ग जैसा ही सोचें।,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा मार्ग जैसा ही सोचें।,स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई क ि ं ग | ल ं ब ी | प ै द ल | य ा त ् र ा | म ा र ् ग | ज ै स ा | ह ी | स ो च े ं । |,149760,FEMALE
0,1839,18194643634199893480.wav,स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा)...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई ...,138240,FEMALE
1,1926,14585676639063535081.wav,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,म न ो व ि ज ् ञ ा न | स ह ि त | व ि ज ् ञ ा न ...,106560,MALE
2,1926,9520717118868623421.wav,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,म न ो व ि ज ् ञ ा न | स ह ि त | व ि ज ् ञ ा न ...,132480,FEMALE
3,1859,13603726120458761913.wav,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,म ध ् य | य ु ग | क े | अ ं त | म े ं | प श ् ...,297600,FEMALE
4,1859,3897753980006371820.wav,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,म ध ् य | य ु ग | क े | अ ं त | म े ं | प श ् ...,222720,FEMALE


In [ ]:
test_df = pd.read_csv(test_tsv_path, sep="\t", header=None)

print("Shape:", test_df.shape)
test_df.head()

Shape: (418, 7)


,0,1,2,3,4,5,6
0,1839,7251883826251297781.wav,स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा)...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई ...,149760,FEMALE
1,1839,18194643634199893480.wav,स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा)...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई ...,138240,FEMALE
2,1926,14585676639063535081.wav,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,म न ो व ि ज ् ञ ा न | स ह ि त | व ि ज ् ञ ा न ...,106560,MALE
3,1926,9520717118868623421.wav,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,म न ो व ि ज ् ञ ा न | स ह ि त | व ि ज ् ञ ा न ...,132480,FEMALE
4,1859,13603726120458761913.wav,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,म ध ् य | य ु ग | क े | अ ं त | म े ं | प श ् ...,297600,FEMALE


In [ ]:
test_df.columns = [
    "id",
    "filename",
    "raw_transcription",
    "transcription",
    "char_tokens",
    "num_samples",
    "gender",
]

print(test_df.columns.tolist())
test_df.head()

['id', 'filename', 'raw_transcription', 'transcription', 'char_tokens', 'num_samples', 'gender']


,id,filename,raw_transcription,transcription,char_tokens,num_samples,gender
0,1839,7251883826251297781.wav,स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा)...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई ...,149760,FEMALE
1,1839,18194643634199893480.wav,स्कीइंग मार्ग को एक हाईकिंग (लंबी पैदल यात्रा)...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स ् क ी इ ं ग | म ा र ् ग | क ो | ए क | ह ा ई ...,138240,FEMALE
2,1926,14585676639063535081.wav,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,म न ो व ि ज ् ञ ा न | स ह ि त | व ि ज ् ञ ा न ...,106560,MALE
3,1926,9520717118868623421.wav,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,म न ो व ि ज ् ञ ा न | स ह ि त | व ि ज ् ञ ा न ...,132480,FEMALE
4,1859,13603726120458761913.wav,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,म ध ् य | य ु ग | क े | अ ं त | म े ं | प श ् ...,297600,FEMALE


In [ ]:
audio_tar_path = hf_hub_download(
    repo_id="google/fleurs",
    filename="data/hi_in/audio/test.tar.gz",
    repo_type="dataset"
)

print("Downloaded audio tar path:")
print(audio_tar_path)

data/hi_in/audio/test.tar.gz:   0%|          | 0.00/249M [00:00<?, ?B/s]

Downloaded audio tar path:
/root/.cache/huggingface/hub/datasets--google--fleurs/snapshots/d7c758a6dceecd54a98cac43404d3d576e721f07/data/hi_in/audio/test.tar.gz


In [ ]:
import tarfile

FLEURS_AUDIO_DIR = "/content/fleurs_hi_test_audio"
os.makedirs(FLEURS_AUDIO_DIR, exist_ok=True)

with tarfile.open(audio_tar_path, "r:gz") as tar:
    tar.extractall(FLEURS_AUDIO_DIR)

print("Extracted to:", FLEURS_AUDIO_DIR)
print("Number of extracted files:", len(os.listdir(FLEURS_AUDIO_DIR)))

/tmp/ipykernel_3364/4039066979.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(FLEURS_AUDIO_DIR)


Extracted to: /content/fleurs_hi_test_audio
Number of extracted files: 1


In [ ]:
for root, dirs, files in os.walk(FLEURS_AUDIO_DIR):
    print("ROOT:", root)
    print("DIRS:", dirs[:10])
    print("FILES:", files[:10])
    print("-" * 60)

ROOT: /content/fleurs_hi_test_audio
DIRS: ['test']
FILES: []
------------------------------------------------------------
ROOT: /content/fleurs_hi_test_audio/test
DIRS: []
FILES: ['18050109654890999214.wav', '18229547637409880012.wav', '6716012942964313180.wav', '5950668010832719037.wav', '18245634017773510582.wav', '2721526147232986835.wav', '13717677002215321761.wav', '8784492922097846920.wav', '17191669298005207755.wav', '16674359696770797961.wav']
------------------------------------------------------------


In [ ]:
FLEURS_WAV_DIR = os.path.join(FLEURS_AUDIO_DIR, "test")

print("Wav dir:", FLEURS_WAV_DIR)
print("Sample files:", os.listdir(FLEURS_WAV_DIR)[:10])

Wav dir: /content/fleurs_hi_test_audio/test
Sample files: ['18050109654890999214.wav', '18229547637409880012.wav', '6716012942964313180.wav', '5950668010832719037.wav', '18245634017773510582.wav', '2721526147232986835.wav', '13717677002215321761.wav', '8784492922097846920.wav', '17191669298005207755.wav', '16674359696770797961.wav']


In [ ]:
test_df["audio_path"] = test_df["filename"].apply(lambda x: os.path.join(FLEURS_WAV_DIR, x))

print("All files exist:", test_df["audio_path"].apply(os.path.exists).all())
test_df[["filename", "audio_path", "transcription"]].head()

All files exist: True


,filename,audio_path,transcription
0,7251883826251297781.wav,/content/fleurs_hi_test_audio/test/72518838262...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...
1,18194643634199893480.wav,/content/fleurs_hi_test_audio/test/18194643634...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...
2,14585676639063535081.wav,/content/fleurs_hi_test_audio/test/14585676639...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...
3,9520717118868623421.wav,/content/fleurs_hi_test_audio/test/95207171188...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...
4,13603726120458761913.wav,/content/fleurs_hi_test_audio/test/13603726120...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...


In [ ]:
sample_df = test_df.head(20).copy()
print("Sample size:", len(sample_df))
sample_df[["filename", "audio_path", "transcription"]].head()

Sample size: 20


,filename,audio_path,transcription
0,7251883826251297781.wav,/content/fleurs_hi_test_audio/test/72518838262...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...
1,18194643634199893480.wav,/content/fleurs_hi_test_audio/test/18194643634...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...
2,14585676639063535081.wav,/content/fleurs_hi_test_audio/test/14585676639...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...
3,9520717118868623421.wav,/content/fleurs_hi_test_audio/test/95207171188...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...
4,13603726120458761913.wav,/content/fleurs_hi_test_audio/test/13603726120...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...


In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.to(device)

print("Model loaded on:", device)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded on: cuda


In [ ]:
import librosa
import evaluate
from tqdm import tqdm

wer_metric = evaluate.load("wer")

predictions = []
references = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    audio_path = row["audio_path"]
    reference = str(row["transcription"]).strip()

    audio, sr = librosa.load(audio_path, sr=16000)

    input_features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    predicted_ids = model.generate(input_features)
    prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

    predictions.append(prediction)
    references.append(reference)

baseline_wer = wer_metric.compute(predictions=predictions, references=references)

results_df = sample_df[["filename", "audio_path", "transcription"]].copy()
results_df["prediction"] = predictions

print("Baseline WER on 20-sample Hindi FLEURS subset:", baseline_wer)
results_df.head()

  0%|          | 0/20 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cust

Baseline WER on 20-sample Hindi FLEURS subset: 0.739406779661017


,filename,audio_path,transcription,prediction
0,7251883826251297781.wav,/content/fleurs_hi_test_audio/test/72518838262...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,श्कीं मार्को एक हाएकिं लंबी पैदल याट्रा मार्क ...
1,18194643634199893480.wav,/content/fleurs_hi_test_audio/test/18194643634...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,अगर बादा बादा बादा बादा बादा बादा बादा बादा बा...
2,14585676639063535081.wav,/content/fleurs_hi_test_audio/test/14585676639...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनु विज्यान सहित विज्यान के सबी माबलो पर अर्स्...
3,9520717118868623421.wav,/content/fleurs_hi_test_audio/test/95207171188...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनो विग्यान सहित विग्यान के सभी मामलों पर आरस्...
4,13603726120458761913.wav,/content/fleurs_hi_test_audio/test/13603726120...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मदेयुक के अंथ में पश्छिम यूरोप ने अपनी श्याली ...


In [ ]:
baseline_csv_path = os.path.join(OUTPUT_DIR, "baseline_fleurs_hi_20_predictions.csv")
results_df.to_csv(baseline_csv_path, index=False)

wer_txt_path = os.path.join(OUTPUT_DIR, "baseline_fleurs_hi_20_wer.txt")
with open(wer_txt_path, "w", encoding="utf-8") as f:
    f.write(f"Baseline WER on 20-sample Hindi FLEURS subset: {baseline_wer}\n")

print("Saved predictions to:", baseline_csv_path)
print("Saved WER to:", wer_txt_path)

Saved predictions to: /content/drive/MyDrive/JoshTalks_ASR/05_outputs/baseline_eval/baseline_fleurs_hi_20_predictions.csv
Saved WER to: /content/drive/MyDrive/JoshTalks_ASR/05_outputs/baseline_eval/baseline_fleurs_hi_20_wer.txt


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
